# LLMOpsFoundations

**Module 14 · Lesson 14.1**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- What LLMOps Actually Is
- Non-Determinism Breaks Single-Point Eval
- Cost Is A First-Class Metric, Measured In Tokens
- The Trace Record: The Foundation
- Read Your Traces: Percentiles, Not Averages
- The LLMOps Maturity Model

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

## A demo takes a week; production takes a year

> 💡 **Info**
>
> You can build an LLM feature in an afternoon: call the API, paste in a prompt, ship the demo. Then it goes to production and the real work begins — work that mostly did not exist in the machine-learning operations (MLOps) playbook. **LLMOps** is the discipline of running LLM-powered applications in production, and its defining fact is that **you consume a model instead of training one**. That single change deletes most of MLOps (there is no training pipeline, no feature store, no model registry of your own weights, no retraining schedule) and adds a new set of problems that MLOps never had to solve: you **version prompts** instead of weights; you **evaluate open-ended prose** with judges and golden sets because you cannot run a single accuracy score on it; you **track token cost** per request because it swings a hundredfold; you **trace** every call end to end; and you guard against **prompt injection and PII leaks** that look like ordinary text. The one-line definition to memorise: *LLMOps = MLOps minus training, plus prompts, plus subjective eval, plus token cost.* This lesson makes each of those concrete and hands you the foundational artifact — a structured trace record — that every later lesson in the module builds on.

### 🎯 What you will be able to do after this lesson

- **Build** the four foundational LLMOps tools as runnable models — a structured trace record, a trace analyzer that computes percentiles and cost, a maturity-model scorer, and an investment-decision function.

- **Compare** MLOps with LLMOps, a single-run score with a distribution, and a mean with its percentiles — and say why each difference matters.

- **Analyze** a day of your own traces: total cost, per-feature cost attribution, p50 and p95 latency, and error rate.

- **Evaluate** how much LLMOps a project needs by matching the investment to the stakes and scale, not to enthusiasm.

> 📦 **Info**
>
> ✅ Before you startThis lesson leans on three things you have already met. In **7.6** you first traced an LLM call; here the trace record becomes the foundation of everything. In **12.8** you learned that a healthy mean can hide an ugly p95; here you compute that on your own request logs. And across **Module 13** you priced LLM calls in tokens; here token cost becomes a thing you must *capture* before you can ever cut it. Prompt versioning, evaluation, and serving each get their own lesson later in this module.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🏭 **Analogy**
>
> If **MLOps is the assembly line** that builds and ships a model, **LLMOps is that assembly line plus a quality lab, a cost accountant, and a safety inspector** — all bolted on. Most of what MLOps spent a decade perfecting (training pipelines, model registries, retraining triggers) simply does not apply when you buy the engine from Anthropic, OpenAI, or Google and drop it into your product. But four new departments open the moment you do: someone has to judge whether the prose output is any good (the quality lab), someone has to watch the per-request token bill (the cost accountant), someone has to catch prompt injection and leaked PII (the safety inspector), and someone has to keep a black-box recording of every call (the trace). **Where it breaks down:** unlike a factory you cannot retool the engine itself — it is someone else’s weights, and they can change it under you — so all your leverage is in the departments around it.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“LLMOps is just MLOps with a new name.”** When you consume an API, the training-centric half of MLOps is moot, and the operational work that remains (prompts, subjective eval, token cost, tracing, guardrails) has almost no shared infrastructure with it. Same word “ops”, different discipline.
> - **“It passed the eval once, so it works.”** An LLM at any non-zero temperature gives a different answer each run. One score is a coin flip; you judge a *distribution* over several runs, never a single point.

> 💡 **Info**
>
> ⚠️ Anti-patternTwo ways teams get the investment wrong. **Over-building a demo** — wiring a fifty-user internal tool with continuous-eval, canary rollouts, and multi-region dashboards, and burning weeks of engineering on infrastructure the stakes never justified. And **under-building a product** — shipping a fifty-thousand-user customer feature on print statements and hope, with no traces, no eval, and no idea what the model said when a customer complains. Match the ops to the stakes; both extremes are failures.

---

## 🎯 Concept 1: What LLMOps Actually Is

### What LLMOps Actually Is

LLMOps is the discipline of running LLM apps in production. Its defining move is that you consume a model via API instead of training one - so most MLOps machinery (training, feature store, model registry, retraining) drops away, and a new set of practices (prompt versioning, subjective eval, token-cost logging, tracing, guardrails) appears with no MLOps equivalent.

Start with a clean definition. **LLMOps** is running LLM-powered applications in production: versioning what makes the app behave, evaluating quality continuously, observing every request, controlling cost and safety, and iterating with the same discipline you give backend code. The reason it is a *separate* field from MLOps comes down to one move — **you consume a pre-trained model through an API instead of training your own**. The moment you do, the training-centric half of MLOps becomes irrelevant: there is no training pipeline, no feature store, no registry of *your* weights, no retraining schedule. What is left is a small shared core (you still deploy, still run CI, still watch dashboards) plus a set of genuinely new practices that MLOps never had a reason to build: **prompt versioning** (the prompt is now the thing you change), **evaluation by judge** (the output is prose, so “correct” is subjective), **token-cost logging**, **request tracing**, and **guardrails**. The block counts the three sets, keyless.

> 🏭 **Analogy**
>
> It is **the assembly line plus a quality lab, a cost accountant, and a safety inspector**. MLOps built the line that manufactures a model; when you rent the model instead, the manufacturing half goes dark, but four new departments switch on around the thing you rented — a lab that judges whether its prose output is good, an accountant watching the token bill, an inspector guarding against injection and leaks, and a recorder keeping a black-box log of every call. You did not build the engine, so all your operational work is in the departments surrounding it.

You move a text classifier from a home-grown model to a hosted LLM API. Of the classic MLOps practices, how many carry over unchanged?

**📝 `01_mlops_vs_llmops.py`** - *runnable*

In [ ]:
# LLMOps is not MLOps renamed. When you CONSUME a model via API instead of TRAINING one, most MLOps
# machinery drops away and a new set of operational problems appears. Model it as three sets of practices.
shared     = {"deploy to prod / serving", "CI-CD pipeline", "monitoring dashboards + alerts"}
mlops_only = {"training pipeline", "feature store", "model registry (your weights)", "scheduled retraining"}
llmops_only= {"prompt versioning", "LLM-as-judge eval", "token-cost logging", "request tracing", "guardrails / safety"}
print("Shared ops practices ({}):".format(len(shared)))
for x in sorted(shared):     print("   =", x)
print("Dropped when you consume an API ({}) - no equivalent needed:".format(len(mlops_only)))
for x in sorted(mlops_only): print("   -", x)
print("Added by LLMs ({}) - NONE has an MLOps equivalent:".format(len(llmops_only)))
for x in sorted(llmops_only):print("   +", x)
print()
print("So LLMOps = MLOps  minus training  plus prompts  plus subjective eval  plus token cost.")
print("Only {} practices carry over; the {} that make an LLM app behave are all new. That is why it is its own field.".format(len(shared), len(llmops_only)))

# Output:
# Shared ops practices (3):
#    = CI-CD pipeline
#    = deploy to prod / serving
#    = monitoring dashboards + alerts
# Dropped when you consume an API (4) - no equivalent needed:
#    - feature store
#    - model registry (your weights)
#    - scheduled retraining
#    - training pipeline
# Added by LLMs (5) - NONE has an MLOps equivalent:
#    + LLM-as-judge eval
#    + guardrails / safety
#    + prompt versioning
#    + request tracing
#    + token-cost logging
#
# So LLMOps = MLOps  minus training  plus prompts  plus subjective eval  plus token cost.
# Only 3 practices carry over; the 5 that make an LLM app behave are all new. That is why it is its own field.

- Only a handful of ops practices are **shared** — deploy/serving, CI, and monitoring dashboards.
- Several **drop away** when you consume an API — the training pipeline, feature store, model registry, and retraining.
- And several are **new** — prompt versioning, LLM-as-judge eval, token-cost logging, tracing, guardrails.
- None of the new set has an MLOps equivalent — which is exactly why LLMOps is its own discipline.

#### 💡 What just happened

⚡ What just happened?LLMOps = MLOps minus training, plus prompts, plus subjective eval, plus token cost. Consuming an API deletes the training-centric half of MLOps and adds five practices with no MLOps equivalent. Framing: memorise that one-line definition; it answers most of what follows. Next: the biggest of those new problems - non-determinism.

- The MLOps pipeline: 3 shared practices plus 4 training-only ones
- Toggle ‘consume an API’: the 4 training practices drop out and 5 new LLMOps practices light up

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Non-Determinism Breaks Single-Point Eval

### Non-Determinism Breaks Single-Point Eval

The number-one way LLMs break MLOps: with a fixed seed, classical ML gives one identical output, so a single eval suffices. Above zero temperature the same prompt gives different outputs every run - so a single score is a coin flip. You evaluate a distribution: mean and spread and pass-rate over N runs.

The first and sharpest break is **non-determinism**. A classical model with a fixed seed produces the *same* output every time, so you can evaluate it once and trust the number. An LLM at any temperature above zero produces a *different* output on the same input each run — different wording, and often a different quality score from your judge. That single fact breaks the MLOps habit of comparing a single prediction to a single label. If you run your prompt against one golden example and it scores well, you have learned almost nothing: the very next run might score much lower. The fix is to treat an LLM eval as a **statistical** exercise — run it several times and report the **mean, the spread, and the pass-rate** at your quality bar, not a single point. The block runs the same input several times and shows how wide the scores spread, keyless. (Building real eval suites — judges, golden sets, calibration — is the whole of Lesson 14.3.)

> 👷 **Analogy**
>
> It is **asking eight chefs to cook the same order**. Give a machine the same recipe and it stamps out an identical dish every time — that is classical ML with a fixed seed. Give the order to eight skilled human chefs and you get eight *slightly* different plates: all recognisably the dish, but varying in seasoning and plating. If you judged the kitchen by tasting one plate, you might call it brilliant or mediocre by luck. You have to taste several and talk about the *average* and the *consistency* — which is exactly how you evaluate a non-deterministic model.

You run the SAME prompt on the SAME input eight times with temperature above zero, and score each with a judge. What do you expect?

**📝 `02_non_determinism.py`** - *runnable*

In [ ]:
# The #1 way LLMs break MLOps: NON-DETERMINISM. In classical ML a fixed seed gives one identical output, so a
# single eval suffices. With temperature > 0 the SAME prompt+input gives different outputs - so you eval statistically.
scores = [0.82, 0.79, 0.88, 0.71, 0.85, 0.80, 0.90, 0.77]  # SAME prompt+input, judged 8 times (quality 0..1), illustrative
n = len(scores)
mean = sum(scores) / n
std = (sum((s - mean) ** 2 for s in scores) / n) ** 0.5
threshold = 0.80
passes = sum(1 for s in scores if s >= threshold)
print("Same prompt, same input, run {} times (temperature > 0):".format(n))
print("  scores span {:.2f} to {:.2f}  |  mean {:.3f}  |  std {:.3f}".format(min(scores), max(scores), mean, std))
print("  pass-rate at a {:.2f} bar: {}/{} = {:.0%}".format(threshold, passes, n, passes / n))
print()
print("A SINGLE run could have reported {:.2f} (looks failing) or {:.2f} (looks great) - a coin flip.".format(min(scores), max(scores)))
print("So an LLM eval is a DISTRIBUTION, not a point: report mean +/- std and pass-rate over N runs, never one call.")
print("Classical ML with a fixed seed reproduces one output, so one eval is enough. That assumption is gone here.")

# Output:
# Same prompt, same input, run 8 times (temperature > 0):
#   scores span 0.71 to 0.90  |  mean 0.815  |  std 0.058
#   pass-rate at a 0.80 bar: 5/8 = 62%
#
# A SINGLE run could have reported 0.71 (looks failing) or 0.90 (looks great) - a coin flip.
# So an LLM eval is a DISTRIBUTION, not a point: report mean +/- std and pass-rate over N runs, never one call.
# Classical ML with a fixed seed reproduces one output, so one eval is enough. That assumption is gone here.

- The same prompt, run several times, produces scores spread across a range — not one repeatable number.
- The honest summary is the **mean and the standard deviation**, plus the **pass-rate** at your quality bar.
- A single run could have landed near the bottom (looks failing) or the top (looks great) — a coin flip.
- Classical ML with a fixed seed reproduces one output, so one eval is enough; that assumption is gone here.

#### 💡 What just happened

⚡ What just happened?An LLM at non-zero temperature gives a different answer each run, so a single eval score is a coin flip; you report a distribution (mean, spread, pass-rate over N runs). Tradeoff: eval costs N calls instead of one, in exchange for a number you can trust. Next: the other first-class break - cost, measured in tokens.

- Eight score dots for the same prompt scatter between a low and high around a mean band
- A movable pass-threshold sets the pass-rate; a single deterministic dot contrasts classical ML

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Cost Is A First-Class Metric, Measured In Tokens

### Cost Is A First-Class Metric, Measured In Tokens

In MLOps cost was compute hours; in LLMOps it is a first-class metric measured in tokens, and it swings enormously per request. The same model and feature can cost a hundred times more on one call than another, driven mostly by how much context is in the input - so you must capture input and output tokens on every call.

The second first-class break is **cost**. In MLOps cost meant compute hours and was fairly predictable. In LLMOps cost is measured in **tokens** — input plus output, each at its own price — and it varies wildly from one request to the next. A short classification call and a retrieval-augmented answer that stuffs a long context into the prompt can differ by **about a hundred times** in cost, on the *same* model and the *same* feature, purely because of token count (and the input context usually dominates). The operational consequence is simple and non-negotiable: **you must record input and output tokens on every call**. If your trace does not carry the token counts, you cannot see spend, cannot attribute it to a feature or a user, and cannot cut it later. The block prices two requests and shows the spread, keyless. (Actually *reducing* the bill — caching, routing, prompt trimming — is Module 13 and Lesson 14.5; here the job is only to *capture* it.)

> 🚖 **Analogy**
>
> It is **a taxi meter that charges by the word**. Two riders take the same route in the same cab, but one sits in silence and the other dictates a novel the whole way — and the wordy rider pays a fortune, because the meter counts words, not distance. An LLM call is billed the same way: not by the “trip” but by the tokens, and a request that stuffs a huge retrieved context into the prompt is the passenger who will not stop talking. If you are not watching the meter on every ride, you will be stunned by the monthly bill.

A short classify call sends about fifty input tokens; a retrieval-augmented call stuffs in several thousand tokens of context. Roughly how do their costs compare?

**📝 `03_token_cost.py`** - *runnable*

In [ ]:
# Cost is a FIRST-CLASS metric in LLMOps (it never was in MLOps). The unit is the TOKEN, and the SAME feature on
# the SAME model can cost 100x more on one request than another - so you must LOG tokens per call or you fly blind.
PRICE_IN, PRICE_OUT = 3.00, 15.00                      # $ per 1M tokens, illustrative sonnet-class rates
def cost(in_tok, out_tok):
    return (in_tok * PRICE_IN + out_tok * PRICE_OUT) / 1_000_000
requests = [("short classify", 50, 8), ("RAG answer (retrieved context stuffed in)", 8000, 400)]
costs = {}
for name, i, o in requests:
    c = cost(i, o); costs[name] = c
    print("  {:<42} {:>5} in + {:>4} out tokens -> ${:.6f}".format(name, i, o, c))
ratio = costs["RAG answer (retrieved context stuffed in)"] / costs["short classify"]
print()
print("Same model, same feature: the RAG request costs about {:.0f}x the classify request - all from tokens.".format(ratio))
print("The retrieved context (input) dominates. If a trace does not record input_tokens and output_tokens per call,")
print("you cannot see spend, cannot attribute it, cannot cut it. Logging the tokens comes first; optimising comes later (M13, 14.5).")

# Output:
#   short classify                                50 in +    8 out tokens -> $0.000270
#   RAG answer (retrieved context stuffed in)   8000 in +  400 out tokens -> $0.030000
#
# Same model, same feature: the RAG request costs about 111x the classify request - all from tokens.
# The retrieved context (input) dominates. If a trace does not record input_tokens and output_tokens per call,
# you cannot see spend, cannot attribute it, cannot cut it. Logging the tokens comes first; optimising comes later (M13, 14.5).

- Cost is **tokens times price** — input and output each priced separately, per million tokens.
- The retrieval-augmented request costs roughly a hundred times the short classify request, on the same model.
- The **input** (the retrieved context) dominates — output is the smaller share here.
- So a trace must record input and output tokens per call, or you are blind to spend and cannot attribute it.

#### 💡 What just happened

⚡ What just happened?Cost is a first-class LLMOps metric measured in tokens, and it swings about a hundredfold per request, driven mostly by input context. Tradeoff / rule: log input and output tokens on every call - capturing cost comes before cutting it. Next: the artifact that carries the tokens (and everything else) - the trace record.

- Two requests as token bars; their cost bars sit about a hundredfold apart
- An input-token slider swings the cost bar to show the context dominates

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: The Trace Record: The Foundation

### The Trace Record: The Foundation

The one artifact everything else builds on: a single instrumented call that emits a structured JSON trace record - call id, model, prompt version, input and output tokens, cost, latency, error. Its field names map to the OpenTelemetry GenAI semantic conventions, the 2026 vendor-neutral standard, so any backend can read it. Every tracing tool is this record plus a UI.

Everything so far — token cost, statistical eval, debugging a complaint — needs the same thing underneath it: a **record of what happened on each call**. That record is the foundational LLMOps artifact, and it is smaller than people expect. One instrumented call emits one **structured JSON line** carrying a unique `call_id`, the user and feature, the **model** and **prompt version**, the **input and output token counts**, the **cost**, the **latency**, and any **error**. Build this once and ship it everywhere and you are already past most teams in production. In 2026 the field names have a standard to follow: the **OpenTelemetry GenAI semantic conventions** define vendor-neutral attributes like `gen_ai.request.model` and `gen_ai.usage.input_tokens`, so a trace from Anthropic, OpenAI, or Google looks the same to any backend. The heavyweight tools you will meet in Lesson 14.3 — LangSmith, Langfuse, Phoenix — are *this record* plus a UI, aggregation, and search. The block builds one trace, keyless (the token counts come off the response object; here they are given, so it runs with no key).

> ✈️ **Analogy**
>
> It is **a flight data recorder**. An aircraft’s black box writes one compact, structured entry for every moment of the flight — altitude, speed, control inputs, warnings — so that when anything happens, investigators can replay exactly what occurred. A trace record is the black box for an LLM call: one structured entry per request with the model, the prompt version, the tokens, the cost, the latency, and any error. When a user complains at three in the afternoon, the trace is what lets you answer “what did the model actually see and say?” instead of shrugging.

What is the minimum you should capture on every LLM call to claim real observability?

**📝 `04_trace_record.py`** - *runnable*

In [ ]:
# The foundational LLMOps artifact: one INSTRUMENTED call that emits a structured trace record. Field names map to
# the OpenTelemetry GenAI semantic conventions (the 2026 vendor-neutral standard) so any backend can read them.
import json
PRICE = {"claude-sonnet-4-6": {"in": 3.00, "out": 15.00}}   # illustrative $/1M
def make_trace(call_id, ts, user_id, feature, model, prompt_version, in_tok, out_tok, latency_ms, error, prompt, output):
    p = PRICE.get(model, {"in": 0, "out": 0})
    return {
        "call_id": call_id, "ts": ts, "user_id": user_id, "feature": feature,
        "gen_ai.request.model": model, "prompt_version": prompt_version,
        "gen_ai.usage.input_tokens": in_tok, "gen_ai.usage.output_tokens": out_tok,
        "cost_usd": round((in_tok * p["in"] + out_tok * p["out"]) / 1_000_000, 6),
        "latency_ms": latency_ms, "error": error,
        "prompt_preview": prompt[:50], "output_preview": (output or "")[:50],
    }
# token counts + latency come off the response object; here they are given, so this runs with no network or key.
trace = make_trace("c-0001", "2026-07-10T14:03:07Z", "u_421", "ticket-classifier",
                   "claude-sonnet-4-6", "clf-v3", 62, 5, 240, None,
                   "Ticket: pizza arrived cold, refund please", "refund")
print(json.dumps(trace, indent=2))
print("# One JSON line per call, appended to a .jsonl file. THIS record is the whole foundation of LLM observability.")
print("# LangSmith, Langfuse, and Phoenix are this record plus a UI and aggregation - you meet them in Lesson 14.3.")

# Output:
# {
#   "call_id": "c-0001",
#   "ts": "2026-07-10T14:03:07Z",
#   "user_id": "u_421",
#   "feature": "ticket-classifier",
#   "gen_ai.request.model": "claude-sonnet-4-6",
#   "prompt_version": "clf-v3",
#   "gen_ai.usage.input_tokens": 62,
#   "gen_ai.usage.output_tokens": 5,
#   "cost_usd": 0.000261,
#   "latency_ms": 240,
#   "error": null,
#   "prompt_preview": "Ticket: pizza arrived cold, refund please",
#   "output_preview": "refund"
# }
# # One JSON line per call, appended to a .jsonl file. THIS record is the whole foundation of LLM observability.
# # LangSmith, Langfuse, and Phoenix are this record plus a UI and aggregation - you meet them in Lesson 14.3.

- One call becomes one **JSON line** with everything you need to debug, price, and group it.
- The `gen_ai.*` field names follow the **OpenTelemetry GenAI conventions**, so any backend reads them.
- The **cost** is computed from the token counts and the model’s price — captured, not guessed.
- LangSmith, Langfuse, and Phoenix (Lesson 14.3) are this record plus a UI and aggregation — build it once yourself first.

#### 💡 What just happened

⚡ What just happened?The foundational LLMOps artifact is one instrumented call emitting a structured JSON trace - call id, model, prompt version, tokens, cost, latency, error - with field names following the OpenTelemetry GenAI conventions. Tradeoff: a few lines of wrapper on every call, in exchange for the ability to answer any what-happened question. Next: what the traces tell you once you read them.

- A call flows through and a JSON trace record populates field by field
- The gen_ai.* fields highlight as the OpenTelemetry standard; a token slider recomputes the cost

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Read Your Traces: Percentiles, Not Averages

### Read Your Traces: Percentiles, Not Averages

Once every call emits a trace you become the analyst of your own logs, and the insight is in the percentiles. The mean latency hides both the honest typical (p50) and the painful tail (p95) that an unlucky user actually feels; and a few expensive calls often drive most of the spend. Aggregating the trace file is a core LLMOps skill.

A pile of trace records is only useful if you read it. The instinct is to look at the **average** — average latency, average cost — and the average is exactly what misleads you. The **mean latency** hides two things at once: the **honest typical** request (the median, p50) and the **painful tail** (p95, p99) that an unlucky user actually experiences. A mean can look perfectly healthy while one request in twenty is several times slower. The same is true of cost: a third of your calls can drive nearly all of your spend. So the core skill is to **aggregate the trace file** and compute distributions — total and per-feature cost, error rate, and p50/p95 latency — not single averages. This is the percentile lesson from 12.8 and 13.4, now applied to logs you generated yourself. The block reads a day of traces and computes all of it, keyless.

> 🩺 **Analogy**
>
> It is **a doctor reading the whole chart, not just the average temperature**. A patient whose *average* temperature over a day looks normal might have spiked a dangerous fever every evening — the average smoothed it away. A good clinician looks at the peaks and the spread, not one summary number. Reading traces is the same: the mean latency is the comforting average that hides the evening fever, and the p95 is the spike that tells you which patients are actually suffering.

Your mean latency looks fine, but a few users insist the app is sometimes slow. Which number reveals their experience?

**📝 `05_read_traces.py`** - *runnable*

In [ ]:
# Once every call emits a trace, you become the analyst of your own logs. The insight is in the PERCENTILES, not the
# average: the mean hides both the honest typical (p50) and the painful tail (p95) - and a few calls drive most spend.
classify = [{"lat": l, "cost": 0.00026, "err": None, "feat": "classify"}
            for l in [180,185,190,195,200,205,210,215,220,225,230,235,240]]
rag = [{"lat": l, "cost": 0.030, "err": None, "feat": "rag_answer"}
       for l in [260,280,300,900,950,1100,1300]]
traces = classify + rag
traces[10]["err"] = "rate_limit"                    # one failed call among the 20
n = len(traces)
lats = sorted(t["lat"] for t in traces)
def pct(vals, p):                                   # nearest-rank percentile
    k = max(1, -(-p * len(vals) // 100))            # ceil(p/100 * n)
    return vals[k - 1]
mean_lat = sum(lats) / n
total_cost = sum(t["cost"] for t in traces)
errors = sum(1 for t in traces if t["err"])
rag_cost = sum(t["cost"] for t in traces if t["feat"] == "rag_answer")
print("{} traces:".format(n))
print("  latency  mean {:.0f} ms   p50 {} ms   p95 {} ms   (the mean hides both)".format(mean_lat, pct(lats,50), pct(lats,95)))
print("  errors   {}/{} = {:.0%}".format(errors, n, errors / n))
print("  cost     ${:.4f} total   rag_answer = ${:.4f} ({:.0%} of spend from {:.0%} of calls)".format(
      total_cost, rag_cost, rag_cost / total_cost, sum(1 for t in traces if t["feat"]=="rag_answer") / n))
print()
print("p95 ({} ms) is about {:.0f}x the p50 ({} ms) - that tail is what a user actually feels, and the mean buries it.".format(
      pct(lats,95), pct(lats,95) / pct(lats,50), pct(lats,50)))
print("A third of the calls (RAG) drive almost all the cost. You could not see any of this without the trace records.")

# Output:
# 20 traces:
#   latency  mean 391 ms   p50 225 ms   p95 1100 ms   (the mean hides both)
#   errors   1/20 = 5%
#   cost     $0.2134 total   rag_answer = $0.2100 (98% of spend from 35% of calls)
#
# p95 (1100 ms) is about 5x the p50 (225 ms) - that tail is what a user actually feels, and the mean buries it.
# A third of the calls (RAG) drive almost all the cost. You could not see any of this without the trace records.

- The **mean latency** sits above most requests yet below the tail — it hides both ends.
- The **p50** is the honest typical request; the **p95** is several times larger — the slow experience users feel.
- A small fraction of calls (the retrieval-augmented ones) drives almost all of the **cost**.
- You could not see any of this without the per-call trace records from the previous step.

#### 💡 What just happened

⚡ What just happened?Reading your traces is a percentile skill: the mean latency hides both the honest typical (p50) and the painful tail (p95), and a few calls often drive most of the cost. Tradeoff: percentiles need the full trace file, not a running average, in exchange for seeing what users actually feel. Next: how to grade a whole team’s LLMOps - the maturity model.

- Twenty latency bars with mean, p50, and p95 lines drawn across them
- Drag the tail: p95 jumps while p50 barely moves; a cost split shows a few calls drive most spend

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: The LLMOps Maturity Model

### The LLMOps Maturity Model

A ladder of five levels - Vibes, Logs, Eval, CI, Continuous - each gated by two capabilities. Your level is the highest rung for which every capability up to it is met; you cannot skip a rung. Going from Level 0 (print-and-pray) to Level 1 (structured traces plus prompts in git) is the single biggest jump. Aim one rung up, not four.

Zoom out from one call to a whole team. The **LLMOps maturity model** is a ladder of five levels — **Level 0 Vibes** (print and pray), **Level 1 Logs** (structured traces and prompts in git — the record from the last two steps), **Level 2 Eval** (a golden set, run on every change), **Level 3 CI** (that eval gates every pull request, with cost budgets), and **Level 4 Continuous** (live traffic feeds eval, with auto-rollback). Each rung is gated by a couple of capabilities, and you cannot skip: your true level is the highest rung for which *every* capability up to it is in place. The practical guidance is blunt — **going from Level 0 to Level 1 is the biggest single improvement you will ever make** (structured logging turns “it worked on my laptop” into “here is exactly what happened”), and you should aim *one* rung above where you are, not leap to Level 4. The block scores a sample team and names its next rung, keyless.

> 🦡 **Analogy**
>
> It is **climbing a ladder one rung at a time**. You cannot stand on the fourth rung while your feet have never touched the first three — there is nothing underneath you. A team that tries to build Level-4 continuous evaluation before it has Level-1 structured logs is reaching for a rung with no support below it. And the first step off the ground — from “we print things and hope” to “every call is recorded” — is the one that changes everything; the higher rungs only make sense once you are standing on it.

A team has prompts in git and structured traces, plus a golden set but no automated eval-on-change. Which level are they at, and what is next?

**📝 `06_maturity_model.py`** - *runnable*

In [ ]:
# The LLMOps maturity model: 5 levels, each gated by two capabilities. A team's level is the highest rung for which
# EVERY capability up to it is met - you cannot skip a rung. Going 0 -> 1 is the biggest single jump.
caps = {
    "prompts_in_git": True,  "structured_traces": True,     # Level 1  Logs
    "golden_set": True,      "eval_on_change": False,       # Level 2  Eval
    "ci_eval_gate": False,   "cost_budget": False,          # Level 3  CI
    "live_eval": False,      "auto_rollback": False,        # Level 4  Continuous
}
LEVELS = [(1,"Logs",["prompts_in_git","structured_traces"]),
          (2,"Eval",["golden_set","eval_on_change"]),
          (3,"CI",["ci_eval_gate","cost_budget"]),
          (4,"Continuous",["live_eval","auto_rollback"])]
level, next_gap = 0, None
for num, name, req in LEVELS:
    met = [c for c in req if caps[c]]
    status = "MET" if len(met) == len(req) else "partial ({}/{})".format(len(met), len(req))
    print("  Level {} {:<11} needs {:<40} -> {}".format(num, name, ", ".join(req), status))
    if level == num - 1 and len(met) == len(req):
        level = num
    elif next_gap is None and len(met) < len(req):
        next_gap = (num, name, [c for c in req if not caps[c]])
print()
print("This team is at Level {}. The next dollar buys Level {} ({}): add {}.".format(
      level, next_gap[0], next_gap[1], " + ".join(next_gap[2])))
print("Aim ONE rung up, not four. 0 -> 1 (print-and-pray to structured traces + prompts in git) is the biggest win of all.")

# Output:
#   Level 1 Logs        needs prompts_in_git, structured_traces        -> MET
#   Level 2 Eval        needs golden_set, eval_on_change               -> partial (1/2)
#   Level 3 CI          needs ci_eval_gate, cost_budget                -> partial (0/2)
#   Level 4 Continuous  needs live_eval, auto_rollback                 -> partial (0/2)
#
# This team is at Level 1. The next dollar buys Level 2 (Eval): add eval_on_change.
# Aim ONE rung up, not four. 0 -> 1 (print-and-pray to structured traces + prompts in git) is the biggest win of all.

- Each level is gated by a couple of capabilities; the team’s level is the highest rung fully met.
- This team has Level 1 complete but only part of Level 2, so it sits at **Level 1** — you cannot skip a rung.
- The next dollar buys the missing Level 2 capability (run the eval on every change), not a leap to Level 4.
- Going from Level 0 to Level 1 — print-and-pray to structured traces — is the single biggest jump.

#### 💡 What just happened

⚡ What just happened?The maturity model is a five-rung ladder (Vibes, Logs, Eval, CI, Continuous), each gated by capabilities you cannot skip; your level is the highest fully-met rung. Tradeoff / rule: aim one rung up, and treat 0 to 1 as the biggest win. Next: how far up the ladder a given project should actually climb.

- A five-rung maturity ladder with eight capability toggles
- The ‘you are here’ marker climbs only when every lower rung is fully met

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: Match The Investment To The Stakes

### Match The Investment To The Stakes

Not every project needs the full stack. The honest decision uses scale AND stakes: a hackathon demo needs Level 0; a company-wide internal tool needs Level 1-2; a large customer-facing feature needs the full stack; and a high-stakes domain (medical, legal, financial) needs the full stack plus guardrails and human-in-the-loop regardless of volume. Over-building and under-building are both failures.

Finish with the question that decides how much of all this to build: **how much LLMOps does this project actually need?** The answer is a function of two things — **scale** (users and monthly cost) and **stakes** (what happens when it is wrong). A hackathon demo for a handful of users needs **Level 0**: structured logs to stdout, and move on. A company-wide internal tool needs **Level 1 to 2**: tracing, prompt versioning, and a small golden set. A large customer-facing feature needs the **full stack**: CI eval gates, continuous eval, and alerts. And a **high-stakes** application — medical, legal, financial — needs the full stack *plus* guardrails and human-in-the-loop **regardless of volume**, because the risk, not the traffic, sets the floor. The two failure modes are symmetric: over-building a demo wastes weeks, and under-building a product is negligence. The block runs the decision over four scenarios, keyless.

> 🚒 **Analogy**
>
> It is **right-sizing the fire brigade to the building**. You do not station a full engine company outside a garden shed — and you do not protect a hospital with a single bucket of sand. A garden shed (the hackathon demo) needs almost nothing; an office block (the customer feature) needs a full response plan; and a hospital or a fuel depot (the clinical or financial app) needs the maximum — sprinklers, alarms, and staff on site — even if hardly anyone is inside, because of what burning down would cost. You size the protection to the stakes, not just the square footage.

A clinical triage assistant serves only five hundred users and has a modest bill. How much LLMOps does it need?

**📝 `07_investment_decision.py`** - *runnable*

In [ ]:
# When do you need LLMOps, and how much? Match the investment to the STAKES, not to your enthusiasm. Over-building a
# demo wastes weeks; under-building a product is negligence; high stakes force the full stack regardless of volume.
def recommend(users, monthly_usd, high_stakes):
    if high_stakes:                                   # medical / legal / financial: stakes override volume
        return "FULL stack + guardrails + human-in-the-loop + audit trail"
    if users < 50 and monthly_usd < 50:
        return "Level 0 - structured logs to stdout, move on"
    if users < 5000 and monthly_usd < 5000:
        return "Level 1-2 - tracing + prompt versioning + a small golden set"
    return "Level 3-4 - full stack: CI eval gate + continuous eval + alerts"
scenarios = [("hackathon demo", 8, 20, False),
             ("company-wide internal tool", 2000, 3000, False),
             ("customer-facing feature", 40000, 30000, False),
             ("clinical triage assistant", 500, 4000, True)]
for name, users, spend, stakes in scenarios:
    print("  {:<28} {:>6} users  ${:>6}/mo  stakes={:<5} -> {}".format(name, users, spend, str(stakes), recommend(users, spend, stakes)))
print()
print("The two failure modes: a 50-user demo carrying a full CI-eval stack (wasted), and a 40,000-user feature shipped")
print("on print statements (a fire waiting to happen). And note the clinical bot: LOW volume, but stakes force the full stack.")

# Output:
#   hackathon demo                    8 users  $    20/mo  stakes=False -> Level 0 - structured logs to stdout, move on
#   company-wide internal tool     2000 users  $  3000/mo  stakes=False -> Level 1-2 - tracing + prompt versioning + a small golden set
#   customer-facing feature       40000 users  $ 30000/mo  stakes=False -> Level 3-4 - full stack: CI eval gate + continuous eval + alerts
#   clinical triage assistant       500 users  $  4000/mo  stakes=True  -> FULL stack + guardrails + human-in-the-loop + audit trail
#
# The two failure modes: a 50-user demo carrying a full CI-eval stack (wasted), and a 40,000-user feature shipped
# on print statements (a fire waiting to happen). And note the clinical bot: LOW volume, but stakes force the full stack.

- The decision reads both **scale** (users, monthly cost) and **stakes** (high or not).
- A demo lands at Level 0; a company tool at Level 1–2; a large customer feature at the full stack.
- The clinical bot has *low* volume but **high stakes**, so it jumps straight to the full stack plus guardrails and human review.
- Over-building a demo and under-building a product are the two symmetric failures this rule prevents.

#### 💡 What just happened

⚡ What just happened?Match the LLMOps investment to scale AND stakes: a demo needs Level 0, a company tool Level 1-2, a large customer feature the full stack, and any high-stakes domain the full stack plus guardrails and human-in-the-loop regardless of volume. That is the whole foundation: define it, instrument it, read it, and right-size it.

- Users and monthly-cost sliders plus a high-stakes toggle light up a recommended stack
- Four presets (hackathon, company tool, customer feature, clinical bot) jump to their level

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> LLMOps is the operational discipline for apps that **consume** a model instead of training one (Step 1): the training half of MLOps drops away and five new practices appear. Two of them are first-class breaks — **non-determinism**, which forces statistical eval over single-point scores (Step 2), and **token cost**, which swings a hundredfold per request and must be captured (Step 3). Both are carried by the one foundational artifact, the **structured trace record** with its OpenTelemetry `gen_ai.*` fields (Step 4). Once every call emits a trace, you read the file with **percentiles, not averages** (Step 5), grade the whole effort on the five-rung **maturity ladder** (Step 6), and **right-size the investment to the stakes** (Step 7). Ask two questions of any LLM app: is every call producing a trace, and is the ops investment matched to what happens when it is wrong?

**📦 **Tracing & observability****

| Capability | Representative 2026 tools | Covered in |
|---|---|---|
| Tracing & observability | LangSmith, Langfuse, Arize Phoenix, Helicone, OpenLLMetry | Lesson 14.3 (foundation here) |
| Prompt versioning & A/B | PromptLayer, LangSmith, Latitude, git | Lesson 14.2 |
| Evaluation & judges | Promptfoo, RAGAS, DeepEval, Braintrust | Lesson 14.3 |
| Error analysis & custom evals | trace annotation, failure taxonomies | Lesson 14.4 |
| Serving & cost optimization | vLLM, prompt caching, model routing (Module 13) | Lesson 14.5 |
| Guardrails & CI eval gates | Guardrails AI, Llama Guard, Promptfoo + CI | Lessons 14.3 & 14.6 |

> 📦 **Info**
>
> ➡️ Where this goes nextThe trace record you built here is the raw material for the rest of the module. Prompt versioning and A/B testing come next, in Lesson 14.2. Turning traces into golden sets, LLM-as-judge suites, and CI eval gates is the work of Lesson 14.3. Reading those same traces to build a failure taxonomy and custom evals — error analysis and eval-driven development, the fastest-growing LLMOps skill of 2026 — is Lesson 14.4. Serving and cost optimization, building on Module 13, is Lesson 14.5. The primary references are the [OpenTelemetry GenAI semantic conventions](https://opentelemetry.io/docs/specs/semconv/gen-ai/), the [Langfuse docs](https://langfuse.com/docs), and the [error-analysis guides](https://hamel.dev/blog/posts/evals/) of Hamel Husain and Shreya Shankar.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **LLMOpsFoundations**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-14_1.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-14.1.html` - regenerate this notebook whenever the source HTML is updated.*
